In [ ]:
import os, glob

# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q huggingface_hub scapy

# ── Step 2: Clone latest repo ─────────────────────────────────────────────────
%cd /kaggle/working
!rm -rf sovereign-byte-firewall
!git clone https://github.com/TheSPST/sovereign-byte-firewall.git
%cd sovereign-byte-firewall

# ── Step 3: Download Mamba-2 checkpoint from HuggingFace ─────────────────────
from huggingface_hub import hf_hub_download
os.makedirs('checkpoints', exist_ok=True)
ckpt_path = hf_hub_download(
    repo_id='spst01/sovereign-byte-firewall-mamba2',
    filename='checkpoints/ckpt_mamba2.pt',
    local_dir='.'
)
print(f'Checkpoint: {ckpt_path}')

# ── Step 4: Discover where Kaggle mounted the PCAP dataset ───────────────────
print('\n--- Kaggle /kaggle/input contents ---')
for entry in sorted(os.listdir('/kaggle/input')):
    print(' ', entry)

# Auto-detect the dataset mount directory
candidates = [
    '/kaggle/input/network-intrusion-pcap-archive',
    '/kaggle/input/network-intrusion-pcap-archive/network-intrusion-pcap-archive',
]
# Also scan for any directory that has normal.pcap
for root, dirs, files in os.walk('/kaggle/input'):
    if 'normal.pcap' in files:
        candidates.insert(0, root)
        break

DATA_DIR = None
for c in candidates:
    if os.path.isdir(c) and os.path.exists(os.path.join(c, 'normal.pcap')):
        DATA_DIR = c
        break

if DATA_DIR:
    print(f'\n✅ Found PCAP dataset at: {DATA_DIR}')
    pcaps = sorted(glob.glob(os.path.join(DATA_DIR, '*.pcap')))
    print(f'   PCAPs available: {[os.path.basename(p) for p in pcaps]}')
else:
    print('\n❌ Could not locate normal.pcap — check dataset attachment!')
    raise FileNotFoundError('Network Intrusion PCAP Archive not found under /kaggle/input')

In [ ]:
# ── Step 5: Run Zero-Day Evaluation with real adversarial PCAPs ───────────────
import subprocess, sys

# Use the auto-discovered DATA_DIR
BENIGN_CALIB   = os.path.join(DATA_DIR, 'normal.pcap')
BENIGN_HOLDOUT = os.path.join(DATA_DIR, 'normal2.pcap')
ATTACK_DIR     = DATA_DIR
HOLDOUT_ATTACK = os.path.join(DATA_DIR, '0day.pcap')
OUTPUT_DIR     = '/kaggle/working/results_mamba2_gpu'

print('Running evaluation with:')
print(f'  Benign calibration : {BENIGN_CALIB}')
print(f'  Benign holdout     : {BENIGN_HOLDOUT}')
print(f'  Attack dir         : {ATTACK_DIR}')
print(f'  Holdout attack     : {HOLDOUT_ATTACK}')

cmd = [
    sys.executable, 'evaluate_zero_day.py',
    '--ckpt',                 'checkpoints/ckpt_mamba2.pt',
    '--pcap',                 BENIGN_CALIB,
    '--benign_holdout_pcap',  BENIGN_HOLDOUT,
    '--attack_dir',           ATTACK_DIR,
    '--holdout_attack_pcap',  HOLDOUT_ATTACK,
    '--output_dir',           OUTPUT_DIR,
    '--score_agg',            'topk',
    '--topk_frac',            '0.1',
]
result = subprocess.run(cmd, cwd='/kaggle/working/sovereign-byte-firewall')
if result.returncode != 0:
    print('\n❌ Evaluation failed — check output above for errors.')
else:
    print('\n✅ Evaluation complete!')

In [ ]:
# ── Step 6: Display Metrics and ROC Curve ─────────────────────────────────────
import json
from IPython.display import Image, display

metrics_file = '/kaggle/working/results_mamba2_gpu/metrics.json'
if os.path.exists(metrics_file):
    with open(metrics_file) as f:
        metrics = json.load(f)

    calib  = metrics.get('calibration', {})
    holdout= metrics.get('holdout', {}).get('results', {})
    youden = calib.get('youden', {})
    h_youden = holdout.get('youden', {})

    print('==================================================')
    print('       MAMBA-2 KAGGLE GPU BENCHMARK RESULTS       ')
    print('==================================================')
    print(f"  ROC-AUC Score:              {calib.get('auc', 0.0):.4f}")
    print(f"  Calibration FPR (Youden):   {youden.get('fpr', 0.0):.4f}")
    print(f"  Calibration TPR (Youden):   {youden.get('tpr', 0.0):.4f}")
    print(f"  Holdout Zero-Day DR:        {h_youden.get('holdout_attack_detection_rate', 0.0):.4f}")
    print(f"  Holdout Benign FPR:         {h_youden.get('holdout_benign_false_positive_rate', 0.0):.6f}")
    print('==================================================')
else:
    print('❌ metrics.json not found — evaluation may have failed.')

roc_path = '/kaggle/working/results_mamba2_gpu/roc_curve.png'
if os.path.exists(roc_path):
    display(Image(filename=roc_path))